# Problem 20 — Min Stack

- **LeetCode**: [#155](https://leetcode.com/problems/min-stack/)
- **Difficulty**: Easy
- **Pattern**: **Stack with augmented metadata** — the trick is keeping `min` retrievable in O(1)
- **Target time**: 25-35 min

## Problem
Design a stack that supports `push(val)`, `pop()`, `top()`, and `getMin()` — **all in O(1) time**.

**Operations**
- `push(val)`  — push `val` on top.
- `pop()`      — remove the top element.
- `top()`      — return the top element.
- `getMin()`   — return the minimum element in the stack.

**Example sequence**
```
MinStack ms = new MinStack();
ms.push(-2);    // stack: [-2]
ms.push(0);     // stack: [-2, 0]
ms.push(-3);    // stack: [-2, 0, -3]
ms.getMin();    // → -3
ms.pop();       // stack: [-2, 0]
ms.top();       // → 0
ms.getMin();    // → -2
```

## Why it's hard
If I scan the stack to find the min on every `getMin()` call, that's O(n) per call, not O(1). The challenge is augmenting the stack with extra metadata so the min is **always available at the top** without scanning.

## Two valid approaches — think before coding

**Approach A — Two stacks (cleanest)**
- One main stack for values. One auxiliary stack for *running* mins.
- On push: append `val` to main; append `min(val, current_min)` to min stack.
- On pop: pop both stacks.
- `getMin()` = top of min stack.

**Approach B — Single stack of (val, current_min) tuples**
- Each entry is a pair: the value and the min of the stack *up to and including this entry*.
- Same logic, packed into one stack.

In [20]:
class MinStack:
    def __init__(self):
        self.main_stack=list()
        self.min_stack=list()

    def push(self, val: int) -> None:
        self.main_stack.append(val)
        if (len(self.min_stack) != 0):
            self.min_stack.append(min(val,self.min_stack[-1]))
        else:
            self.min_stack.append(val)

    def pop(self) -> None:
        self.main_stack.pop()
        self.min_stack.pop()

    def top(self) -> int:
        return self.main_stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]


In [21]:
def run(ops):
    # ops is a list of (method_name, args, expected_or_None) tuples.
    # Methods returning None just verify they don't crash.
    ms = MinStack()
    for i, (method, args, expected) in enumerate(ops):
        got = getattr(ms, method)(*args)
        if expected is not None:
            assert got == expected, f'op {i} {method}{tuple(args)}: got {got}, expected {expected}'
        print(f'  {method}{tuple(args)} -> {got}')
    print('OK\n')

# Sample sequence from the problem statement
print('Test 1: classic example')
run([
    ('push',    [-2], None),
    ('push',    [0],  None),
    ('push',    [-3], None),
    ('getMin',  [],   -3),
    ('pop',     [],   None),
    ('top',     [],   0),
    ('getMin',  [],   -2),
])

# Duplicates of the min — make sure pop() doesn't lose the min prematurely
print('Test 2: duplicate mins (the classic gotcha)')
run([
    ('push',    [1],  None),
    ('push',    [1],  None),    # duplicate min
    ('push',    [2],  None),
    ('getMin',  [],   1),
    ('pop',     [],   None),    # pops 2, min is still 1
    ('getMin',  [],   1),
    ('pop',     [],   None),    # pops second 1, min must STILL be 1
    ('getMin',  [],   1),
])

# Single element
print('Test 3: single element')
run([
    ('push',    [42], None),
    ('top',     [],   42),
    ('getMin',  [],   42),
    ('pop',     [],   None),
])

# Increasing sequence — min stays at first push
print('Test 4: monotonically increasing')
run([
    ('push',    [1], None),
    ('push',    [2], None),
    ('push',    [3], None),
    ('getMin',  [],  1),
    ('pop',     [],  None),
    ('getMin',  [],  1),
])

# Decreasing sequence — min should follow the top
print('Test 5: monotonically decreasing')
run([
    ('push',    [3], None),
    ('push',    [2], None),
    ('push',    [1], None),
    ('getMin',  [],  1),
    ('pop',     [],  None),
    ('getMin',  [],  2),
    ('pop',     [],  None),
    ('getMin',  [],  3),
])


Test 1: classic example
  push(-2,) -> None
  push(0,) -> None
  push(-3,) -> None
  getMin() -> -3
  pop() -> None
  top() -> 0
  getMin() -> -2
OK

Test 2: duplicate mins (the classic gotcha)
  push(1,) -> None
  push(1,) -> None
  push(2,) -> None
  getMin() -> 1
  pop() -> None
  getMin() -> 1
  pop() -> None
  getMin() -> 1
OK

Test 3: single element
  push(42,) -> None
  top() -> 42
  getMin() -> 42
  pop() -> None
OK

Test 4: monotonically increasing
  push(1,) -> None
  push(2,) -> None
  push(3,) -> None
  getMin() -> 1
  pop() -> None
  getMin() -> 1
OK

Test 5: monotonically decreasing
  push(3,) -> None
  push(2,) -> None
  push(1,) -> None
  getMin() -> 1
  pop() -> None
  getMin() -> 2
  pop() -> None
  getMin() -> 3
OK



## Pattern & Complexity

| | |
|---|---|
| **Pattern** | Stack with auxiliary structure to keep aggregate (min) at the top in O(1) |
| **Time**    | O(1) per op — push, pop, top, getMin |
| **Space**   | O(n) — both stacks grow with input |

## One-line pattern note
To keep an aggregate (min/max/etc) available in O(1), maintain a parallel stack of running aggregates. Push and pop in lockstep with the main stack.